# SmolVLA Architecture Investigation & Profiling
# SmolVLA inference optimization  
**Joshua Momo | April 2026**

This notebook documents the systematic investigation of SmolVLA's inference pipeline
before writing any optimization code. Every finding here informs the optimization
strategy in Notebook 2.

**Hardware:** Google Colab T4 (16GB). All timing numbers are T4-specific — final A10G benchmarks come from separate Modal runs.

## Questions to Answer
1. Does `predict_action_chunk()` already cache VLM KV across ODE steps, or recompute?
2. What does the postprocessor do? Is it just affine? Can we vectorize?
3. Does the ODE loop have dynamic shapes or control flow that would break `torch.compile`?
4. Where does inference time actually go? (profiling)
5. Is `lm_head` truly dead during action generation?
6. Can `torch.compile` run without graph breaks on the expert?

## 1. Environment Setup

In [ ]:
# Install LeRobot with SmolVLA dependencies
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git"
!pip install -q num2words

In [ ]:
import torch
import numpy as np
import inspect
import textwrap
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Load Model & Inspect Structure

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
policy = policy.to(device).eval()
print("Model loaded.")

In [ ]:
# Top-level module names and param counts
print(f"{'Module':<50} {'Params':>12} {'Size (MB)':>10}")
print("-" * 75)
total = 0
for name, module in policy.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    size_mb = sum(p.numel() * p.element_size() for p in module.parameters()) / 1e6
    print(f"{name:<50} {n_params:>12,} {size_mb:>10.1f}")
    total += n_params
print("-" * 75)
print(f"{'TOTAL':<50} {total:>12,}")

### 2.1 Full Architecture Dump

Print the complete module tree to map every component, its dimensions, and parameter count.
This dump is the ground truth for all optimization decisions — no guessing at shapes.

In [ ]:
# Full architecture dump — shows every layer, dimension, and param shape
print(policy.model)

## 3. Critical Source Code Investigation

The goal is to trace the full call chain from `predict_action_chunk()` down to the
inner ODE loop, answering: what runs once vs. what runs 10x, where are the caching
boundaries, and what computation is redundant.

### 3.1 `predict_action_chunk()` — Top-level entry point

In [ ]:
# Read the actual implementation
src = inspect.getsource(policy.predict_action_chunk)
print(src)

### 3.2 `_get_action_chunk()` — Noise sampling and postprocessing

This is called by `predict_action_chunk`. Need to see where noise is sampled,
how `sample_actions` is called, and what postprocessing happens.

In [ ]:
src = inspect.getsource(policy._get_action_chunk)
print(src)

### 3.3 `sample_actions()` — The ODE loop wrapper

This is where the VLM KV cache is created (once) and the ODE denoising loop runs (10x).
Critical question: does the VLM forward pass run once or inside the loop?

In [ ]:
src = inspect.getsource(policy.model.sample_actions)
print(src)

**Finding: VLM KV cache IS already cached across ODE steps.**

`sample_actions()` calls `vlm_with_expert.forward(fill_kv_cache=True)` ONCE before the loop,
then passes `past_key_values` to all 10 `denoise_step()` calls with `fill_kv_cache=False`.
This is NOT a free optimization win — the baseline already does it correctly.

- VLM forward pass: runs **once** (before ODE loop)
- KV cache: created once, passed to all 10 expert calls
- ODE steps: 10 (configurable via `self.config.num_steps`)
- Noise: sampled once before loop as `x_t ~ N(0, I)` with shape `[B, 50, max_action_dim]`
- Time direction: **1.0 → 0.0** (reverse), `dt = -1.0 / num_steps`

### 3.4 `denoise_step()` — Single ODE step (runs 10x)

Each call = full 16-layer expert transformer pass. This is the dominant cost.
Look for redundant computation that is identical across all 10 calls.

In [ ]:
src = inspect.getsource(policy.model.denoise_step)
print(src)

### 3.5 `embed_suffix()` — Action + time embedding (runs 10x)

Called inside each `denoise_step`. Projects action tokens and time embeddings
into the expert's hidden space. Check for redundant mask/position construction.

In [ ]:
src = inspect.getsource(policy.model.embed_suffix)
print(src)

### 3.6 `embed_prefix()` — VLM feature assembly (runs once per timestep)

Processes images through SigLIP, embeds language tokens, projects robot state.
Check what can be cached within an episode (language tokens don't change).

In [ ]:
src = inspect.getsource(policy.model.embed_prefix)
print(src)

### 3.7 Redundant Computation Analysis

From reading `denoise_step()` and `embed_suffix()`, the following are computed
identically in all 10 ODE steps but only need to be computed once:

| Redundant Operation | What It Computes | Depends On |
|---|---|---|
| `torch.ones(B, 50)` for `suffix_pad_masks` | Literal constant | Nothing |
| `torch.tensor([1]*50)` for `suffix_att_masks` | Literal constant | Nothing |
| `make_att_2d_masks(suffix_pad, suffix_att)` | 2D attention mask | Constants above |
| `prefix_pad_masks[:, None, :].expand(B, 50, prefix_len)` | Cross-attention mask | Fixed prefix |
| `torch.cat([prefix_mask_2d, suffix_mask_2d])` | Full attention mask | Constants above |
| `torch.cumsum(suffix_pad_masks)` for `position_ids` | Position IDs | Constants above |
| `torch.ones(B, 50)` for `action_time_mask` in `embed_suffix` | Literal constant | Nothing |

**What actually changes each step:** only `x_t` (Euler-evolved) and `timestep` (deterministic: 1.0, 0.9, ..., 0.1).

**Implication:** Precomputing these 7 items before the ODE loop eliminates 9 redundant constructions
per inference call. The `aten::cat` and `aten::cumsum` ops appear in profiler results.

### 3.8 Verify Helper Function Imports

Check that key internal functions (`make_att_2d_masks`, `create_sinusoidal_pos_embedding`)
are accessible for potential reimplementation of the ODE loop.

In [ ]:
# Verify helper functions are importable
try:
    from lerobot.policies.smolvla.modeling_smolvla import make_att_2d_masks
    print("make_att_2d_masks: importable")
except ImportError:
    print("make_att_2d_masks: NOT importable (check module path)")

try:
    from lerobot.policies.smolvla.modeling_smolvla import create_sinusoidal_pos_embedding
    print("create_sinusoidal_pos_embedding: importable")
except ImportError:
    print("create_sinusoidal_pos_embedding: NOT importable")

try:
    from lerobot.policies.smolvla.modeling_smolvla import populate_queues
    print("populate_queues: importable")
except ImportError:
    print("populate_queues: NOT importable")

### 3.9 Postprocessor — What Does It Do?

The baseline loops over 50 actions individually through the postprocessor.
If it is just an affine transform, the entire chunk can be processed at once.

In [ ]:
from lerobot.policies.factory import make_pre_post_processors

preprocessor, postprocessor = make_pre_post_processors(
    policy.config,
    pretrained_path="lerobot/smolvla_base",
    preprocessor_overrides={"device_processor": {"device": str(device)}},
    postprocessor_overrides={"device_processor": {"device": str(device)}},
)

print("=== Postprocessor ===")
print(type(postprocessor))
print(postprocessor)

In [ ]:
# Read the postprocessor's __call__ or forward method
try:
    src = inspect.getsource(postprocessor.__call__)
    print(src)
except (TypeError, OSError):
    # May be a composed transform — inspect its components
    print("Cannot inspect __call__ directly. Listing components:")
    if hasattr(postprocessor, 'transforms'):
        for i, t in enumerate(postprocessor.transforms):
            print(f"\n--- Transform {i}: {type(t).__name__} ---")
            print(t)
            try:
                print(inspect.getsource(t.__call__))
            except (TypeError, OSError):
                pass
    elif hasattr(postprocessor, 'steps'):
        for name, step in postprocessor.steps:
            print(f"\n--- Step: {name} ({type(step).__name__}) ---")
            print(step)

**Finding: Postprocessor is effectively identity.**

The `UnnormalizerProcessorStep` applies `action * std + mean`, but the stats are `mean=0, std=1`
for all action dimensions. This means the postprocessor does nothing useful — it is a no-op
affine transform plus a device transfer.

- Postprocessor = affine (scale + offset): **Yes, but mean=0, std=1 (identity)**
- Can be applied to entire chunk at once: **Yes — or skipped entirely**
- Moves data between devices: **Yes (DeviceProcessorStep)**

**Implication:** The baseline's per-action loop (50 Python calls) is pure overhead.
Replace with a single `.cpu().numpy()` on the full `[50, 6]` tensor.

### 3.10 ODE / Flow Matching Configuration

In [ ]:
# Check config for ODE/flow matching parameters
config = policy.config
print("=== Flow Matching / ODE Config ===")
for attr in dir(config):
    if any(kw in attr.lower() for kw in ['step', 'ode', 'flow', 'noise', 'chunk', 'action', 'sample', 'denois']):
        val = getattr(config, attr, None)
        if not callable(val) and not attr.startswith('_'):
            print(f"  {attr}: {val}")

## 4. Dead Weight Verification

From weight surgery (the architecture inspection): `lm_head` has 47.3M params and is never called
during action generation. Verify by checking both the parameter list and the
source code of every function in the inference path.

In [ ]:
# Check if lm_head exists and count its params
lm_head_found = False
for name, param in policy.named_parameters():
    if 'lm_head' in name:
        if not lm_head_found:
            print("=== lm_head parameters ===")
            lm_head_found = True
        print(f"  {name}: {param.shape} ({param.numel():,} params, {param.numel() * param.element_size() / 1e6:.1f} MB)")

if not lm_head_found:
    print("No lm_head found in model parameters.")
else:
    # Verify it's not called during predict_action_chunk
    # Check by searching the source code for 'lm_head'
    src = inspect.getsource(policy.predict_action_chunk)
    if 'lm_head' in src:
        print("\nWARNING: lm_head appears in predict_action_chunk source!")
    else:
        print("\nConfirmed: lm_head NOT referenced in predict_action_chunk.")
        print("Safe to prune for free memory savings.")

In [ ]:
# Also check the VLM forward for lm_head usage
for name, mod in policy.named_children():
    if 'vlm' in name.lower() or 'language' in name.lower() or 'model' in name.lower():
        print(f"=== Checking {name} for lm_head usage ===")
        try:
            src = inspect.getsource(mod.forward)
            if 'lm_head' in src:
                print("lm_head IS used in VLM forward. Need to check if that code path is reached.")
                for i, line in enumerate(src.split('\n')):
                    if 'lm_head' in line:
                        print(f"  Line {i}: {line.strip()}")
            else:
                print("lm_head not referenced in forward().")
        except (TypeError, OSError):
            print(f"  Cannot inspect {name}.forward()")

**Finding: `lm_head` confirmed dead.**

`lm_head` (`Linear(960, 49280)`) = 47.3M params = ~90MB in BF16. It exists in the model
weights but is never called in `predict_action_chunk`, `_get_action_chunk`, `sample_actions`,
or `denoise_step`. Safe to delete at load time for a free memory win.

`del policy.model.vlm_with_expert.vlm.lm_head` saves ~90MB with zero accuracy risk.

## 5. Profiling — Where Does Time Go?

Run forward passes under `torch.profiler` to identify the dominant operations.
All timing here is on **T4** — A10G numbers come from separate Modal benchmarks.

In [ ]:
from lerobot.policies.utils import prepare_observation_for_inference

# Create a realistic dummy input
dummy_obs = prepare_observation_for_inference(
    {
        "observation.state": np.zeros((8,), dtype=np.float32),
        "observation.images.camera1": np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8),
        "observation.images.camera2": np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8),
    },
    device,
    task="pick up the red block",
)
dummy_obs = preprocessor(dummy_obs)
print("Dummy observation keys:", list(dummy_obs.keys()))
for k, v in dummy_obs.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape} {v.dtype}")

In [ ]:
# Warmup passes (important for accurate profiling)
with torch.inference_mode():
    for _ in range(3):
        _ = policy.predict_action_chunk(dummy_obs)
torch.cuda.synchronize()
print("Warmup complete.")

In [ ]:
# Simple wall-clock timing
times = []
with torch.inference_mode():
    for _ in range(10):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = policy.predict_action_chunk(dummy_obs)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)

print(f"predict_action_chunk latency (T4):")
print(f"  Mean: {np.mean(times):.1f} ms")
print(f"  Std:  {np.std(times):.1f} ms")
print(f"  Min:  {np.min(times):.1f} ms")
print(f"  Output shape: {out.shape}")
print(f"  Output dtype: {out.dtype}")

In [ ]:
# Detailed profiling with torch.profiler
from torch.profiler import profile, record_function, ProfilerActivity

with torch.inference_mode():
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        with_stack=True,
    ) as prof:
        _ = policy.predict_action_chunk(dummy_obs)
        torch.cuda.synchronize()

# Top CUDA operations by time
print("=== Top 20 CUDA Operations ===")
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

In [ ]:
# Top operations by CPU time (catches Python overhead)
print("=== Top 20 CPU Operations ===")
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))

In [ ]:
# Export Chrome trace for detailed visualization
prof.export_chrome_trace("smolvla_profile_trace.json")
print("Trace exported to smolvla_profile_trace.json")
print("Download and open at chrome://tracing for flamegraph view.")

## 6. Component-Level Timing

Manually time each stage to confirm the bottleneck hierarchy:
SigLIP vs SmolLM2 vs Expert (x10 ODE steps) vs pre/postprocessing.

All times on **T4** — for reference only.

In [ ]:
# Time preprocessing alone
raw_obs = {
    "observation.state": np.zeros((8,), dtype=np.float32),
    "observation.images.camera1": np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8),
    "observation.images.camera2": np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8),
}

preproc_times = []
for _ in range(20):
    t0 = time.perf_counter()
    obs = prepare_observation_for_inference(raw_obs, device, task="pick up the red block")
    obs = preprocessor(obs)
    t1 = time.perf_counter()
    preproc_times.append((t1 - t0) * 1000)

print(f"Preprocessing: {np.mean(preproc_times):.2f} ms (std: {np.std(preproc_times):.2f})")

# Time postprocessing alone (the baseline's per-action loop)
with torch.inference_mode():
    action_chunk = policy.predict_action_chunk(obs)

if action_chunk.ndim == 2:
    action_chunk = action_chunk.unsqueeze(0)

# Baseline postprocessing: per-action loop
postproc_times = []
for _ in range(20):
    t0 = time.perf_counter()
    processed = []
    for i in range(action_chunk.shape[1]):
        single = action_chunk[:, i, :]
        p = postprocessor(single)
        processed.append(p)
    result = torch.stack(processed, dim=1).squeeze(0)
    _ = result.detach().cpu().numpy()
    t1 = time.perf_counter()
    postproc_times.append((t1 - t0) * 1000)

print(f"Postprocessing (baseline loop): {np.mean(postproc_times):.2f} ms (std: {np.std(postproc_times):.2f})")

# Vectorized postprocessing attempt
vec_times = []
for _ in range(20):
    t0 = time.perf_counter()
    try:
        p = postprocessor(action_chunk.squeeze(0))
        _ = p.detach().cpu().numpy()
        vec_ok = True
    except Exception as e:
        vec_ok = False
        print(f"Vectorized postprocessing failed: {e}")
        break
    t1 = time.perf_counter()
    vec_times.append((t1 - t0) * 1000)

if vec_ok:
    print(f"Postprocessing (vectorized): {np.mean(vec_times):.2f} ms (std: {np.std(vec_times):.2f})")
    print(f"Speedup: {np.mean(postproc_times) / np.mean(vec_times):.1f}x")

## 7. torch.compile Feasibility Check

Test whether `torch.compile` works on the model without graph breaks.

> **T4 caveat:** The T4 GPU lacks native BF16 support. When torch.compile encounters BF16
> operations, it falls back to software-emulated MAGMA kernels (which consumed 75.65% of
> CUDA time in profiling). This means any torch.compile speedup/slowdown measured on T4
> is **invalid for A10G**, which has native BF16 support. The 0.79x "slowdown" observed
> below is a T4 artifact. Must re-test on A10G (done via Modal — see Notebook 2).

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

# Try compiling predict_action_chunk
try:
    compiled_predict = torch.compile(
        policy.predict_action_chunk,
        mode="reduce-overhead",
        fullgraph=False,  # Allow graph breaks initially to see if it works at all
    )
    print("torch.compile succeeded (fullgraph=False).")

    # Run it
    with torch.inference_mode():
        torch.cuda.synchronize()
        # First call triggers compilation
        print("Running first compiled call (triggers compilation)...")
        t0 = time.perf_counter()
        out = compiled_predict(dummy_obs)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        print(f"  First call (compile + run): {(t1-t0)*1000:.0f} ms")

        # Subsequent calls should be faster
        times_compiled = []
        for _ in range(10):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            out = compiled_predict(dummy_obs)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            times_compiled.append((t1-t0)*1000)

        print(f"  Compiled mean: {np.mean(times_compiled):.1f} ms")
        print(f"  vs eager mean: {np.mean(times):.1f} ms")
        print(f"  Speedup: {np.mean(times) / np.mean(times_compiled):.2f}x")

except Exception as e:
    print(f"torch.compile FAILED: {e}")
    print("Will need to compile sub-components individually.")

In [ ]:
# List model children to find expert module name for targeted compilation
print("=== Model children (to find expert module name) ===")
for name, mod in policy.named_children():
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {name}: {mod.__class__.__name__} ({n:,} params)")

**Finding: torch.compile works but T4 results are misleading.**

- `torch.compile(mode='reduce-overhead', fullgraph=False)` succeeds without errors.
- On T4: 0.79x slowdown (INVALID — caused by BF16 software emulation, see caveat above).
- Key fact for A10G: 13,421 kernel launches per inference call were observed in profiling. `torch.compile` should collapse many of these into fused kernels.
- The expert has fixed shapes (50 tokens, 720-dim) — ideal for CUDA graph capture via `reduce-overhead` mode.

## 8. Memory Baseline

In [ ]:
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

with torch.inference_mode():
    _ = policy.predict_action_chunk(dummy_obs)
    torch.cuda.synchronize()

peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak GPU memory during inference: {peak_mem:.3f} GB")
print(f"Baseline peak memory (from eval harness): 0.899 GB")

# Model weight memory
model_mem = sum(p.numel() * p.element_size() for p in policy.parameters()) / 1e9
print(f"Model weights: {model_mem:.3f} GB")
print(f"Activation/KV cache overhead: {peak_mem - model_mem:.3f} GB")

## 9. Precision Check

Verify the model's native dtype and test FP16 conversion.

**Expectation going in:** Model might be FP32 — converting to FP16 would halve memory.
Let's check.

In [ ]:
# Check current dtype
for name, param in list(policy.named_parameters())[:5]:
    print(f"{name}: {param.dtype}")
    break

**Key finding: Model is already BF16, NOT FP32.**

Weights are `torch.bfloat16`. There is no free 2x memory win from "converting FP32 to FP16".
Memory is already 16-bit (~0.9GB for 450M params).

Now test whether `.half()` (FP16 conversion) works:

In [ ]:
# Test FP16 — does .half() work on a BF16 model?
policy_fp16 = policy.half()

# Rebuild dummy obs for FP16
obs_fp16 = {}
for k, v in dummy_obs.items():
    if isinstance(v, torch.Tensor) and v.is_floating_point():
        obs_fp16[k] = v.half()
    else:
        obs_fp16[k] = v

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

try:
    with torch.inference_mode():
        # Warmup
        for _ in range(3):
            _ = policy_fp16.predict_action_chunk(obs_fp16)
        torch.cuda.synchronize()

        # Time
        fp16_times = []
        for _ in range(10):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            out_fp16 = policy_fp16.predict_action_chunk(obs_fp16)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            fp16_times.append((t1-t0)*1000)

    peak_fp16 = torch.cuda.max_memory_allocated() / 1e9
    print(f"FP16 latency: {np.mean(fp16_times):.1f} ms (vs BF16: {np.mean(times):.1f} ms)")
    print(f"FP16 peak memory: {peak_fp16:.3f} GB")
    print(f"Speedup: {np.mean(times)/np.mean(fp16_times):.2f}x")

except Exception as e:
    print(f"FP16 FAILED: {e}")
    print("\n--> .half() causes dtype mismatch (BF16 buffers vs FP16 weights).")
    print("--> Use torch.autocast('cuda', dtype=torch.float16) instead.")

**Finding: `.half()` FAILS with dtype mismatch.**

The model has internal buffers (e.g., rotary embeddings) that remain BF16 when weights are
converted to FP16, causing `"mat1 and mat2 must have the same dtype"` errors.

**Correct approach:** Use `torch.autocast('cuda', dtype=torch.float16)` which wraps
computation and handles mixed dtypes automatically. On T4 (no native BF16), this eliminates
the software-emulated MAGMA kernels that dominated 75% of CUDA time. On A10G (native BF16),
the benefit is smaller but still positive.

## 10. Key Findings Summary

### Architecture Findings
| Question | Answer |
|----------|--------|
| VLM KV cached across ODE steps? | **Yes** — baseline already caches correctly. Not a free win. |
| Postprocessor = affine? | **Yes, but identity** — mean=0, std=1. Effectively a no-op. |
| Postprocessor vectorizable? | **Yes** — or skip entirely, just `.cpu().numpy()` the chunk. |
| lm_head truly dead? | **Yes** — 47.3M params, ~90MB BF16. Never called. Safe to `del`. |
| Number of ODE steps | **10** (configurable via `config.num_steps`) |
| Model native dtype | **BF16** (not FP32 as initially assumed) |
| `.half()` works? | **No** — dtype mismatch. Use `torch.autocast` instead. |
| torch.compile works? | **Yes** (`fullgraph=False`). T4 result invalid (0.79x slowdown from BF16 emulation). |

### Profiling Findings (T4 — directional only, not valid for A10G)
| Metric | Value |
|--------|-------|
| Mean latency (T4) | ~499 ms |
| Kernel launches per call | 13,421 (torch.compile target) |
| Dominant CUDA cost (T4) | BF16 MAGMA emulation (75.65%) — T4-specific artifact |
| Expert runs per call | 10 (ODE steps) — dominant cost on native BF16 hardware |

### Memory Findings
| Metric | Value |
|--------|-------|
| Model weights (BF16) | ~0.9 GB (450M params x 2 bytes) |
| Peak memory (eval harness) | 0.899 GB |
| lm_head size | ~90 MB (47.3M params x 2 bytes) |
| VLM KV cache size | ~2 MB (tiny) |

### Redundant Computation per ODE Step
7 constant tensor constructions repeated 10x but needed only once:
masks (`ones`, `tensor`, `make_att_2d_masks`), position IDs (`cumsum`), attention mask concat.
Sinusoidal time embeddings are also precomputable (deterministic timesteps).

### Implications for Optimization Strategy
1. **No FP32→FP16 memory win** — already BF16. Use autocast for T4 compat, but A10G runs BF16 natively.
2. **Action Expert is the bottleneck** — runs 10x per call. Quantizing or compiling it has 10x leverage over backbone ops.
3. **Free wins available:** `del lm_head` (~90MB), precompute ODE loop constants, vectorize postprocessor, cache language embeddings within episode.
4. **torch.compile must be tested on A10G** — T4 results are meaningless due to BF16 emulation.
5. **ODE step reduction is highest-impact Tier 2 optimization** — 10→5 steps = 2x speedup on the dominant cost, but MSE risk must be measured.
6. **Action chunk caching could be massive** — 50 actions predicted per call, harness calls per-timestep. If caching is legal (README allows internal state), this is up to 50x throughput.